# Base analítica: una fila por llamada

Convierte las 97 transcripciones con rol asignado (`Transcriptions procesadas/transcriptions
con agente y deudor/`) en una tabla plana lista para comparar humanos vs. IA.

**Entrada:** 48 llamadas humanas + 49 de IA, con `rol` (AGENTE / DEUDOR) en cada segmento.
**Salida:** `Base analitica/base_analitica.csv` — 97 filas, una por llamada.

Solo usa la librería estándar de Python: el resultado es reproducible sin instalar nada.

## Qué mide cada bloque de variables

| Bloque | Pregunta que responde | Variables |
|---|---|---|
| **Ritmo** | ¿Se siente natural la conversación? | latencia de respuesta, silencio, turnos, velocidad |
| **Reparto de habla** | ¿Quién domina la llamada? | palabras y tiempo de voz por rol |
| **Conducta del agente** | ¿Cómo persuade? | presión legal, empatía, permisos, muletillas |
| **Oferta y cierre** | ¿Concreta la negociación? | cifra concreta, *time-to-offer*, canal de pago |
| **Protocolo** | ¿Cumple el guion obligatorio? | se identifica, verifica identidad, declara motivo |
| **Deudor** | ¿Qué resistencia aparece? | objeciones léxicas |
| **Medición** | ¿La llamada es auditable? | diarización colapsada, segmentos sin rol |

Las variables de resultado (compromiso de pago, tipo de objeción, manejo de la objeción) **no
están aquí**: son semánticas y se etiquetan aparte con un LLM, igual que se hizo con el rol
agente/deudor.

## Rutas

In [ ]:
import json
import csv
import re
import statistics as st
from pathlib import Path

REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "Transcriptions procesadas").exists():
    REPO_ROOT = REPO_ROOT.parent  # por si se corre desde /Scripts

BASE_DIR = REPO_ROOT / "Transcriptions procesadas" / "transcriptions con agente y deudor"
OUT_DIR = REPO_ROOT / "Base analitica"
OUT_DIR.mkdir(exist_ok=True)

print("Entrada:", BASE_DIR, "| existe:", BASE_DIR.exists())
print("Salida: ", OUT_DIR)

## Cargar las llamadas

In [ ]:
def cargar(grupo):
    """Lee todos los JSON de un grupo (humanos / ia)."""
    return [json.loads(p.read_text(encoding="utf-8"))
            for p in sorted((BASE_DIR / grupo).glob("*.json"))]


LLAMADAS = [("humano", c) for c in cargar("humanos")] + [("ia", c) for c in cargar("ia")]

print("Humanas:", sum(1 for g, _ in LLAMADAS if g == "humano"))
print("IA:     ", sum(1 for g, _ in LLAMADAS if g == "ia"))

## Léxicos

Cada patrón es una **decisión analítica**, no una lista al azar. Se construyeron leyendo
transcripciones de los dos grupos y verificando que capturan lo que dicen medir.

Dos advertencias que condicionan la lectura:

- Los audios están **censurados con un pitido** sobre nombres propios y entidades, así que
  ningún patrón puede depender del nombre de la empresa. Por eso `identificacion` busca la
  *fórmula* de presentación (`le habla`, `le llamo de`), no la marca.
- `objecion` es un **proxy léxico**, no la objeción real: captura la forma típica de negarse
  ("no tengo", "más adelante"), pero se queda corto con objeciones implícitas. Sirve como
  primera aproximación y se validará contra el etiquetado con LLM.

In [ ]:
PATRONES = {
    # Oferta concreta: montos, cuotas o porcentajes de descuento
    "cifra": r"\b\d[\d\.,]*\s*(mil|millon|millones|%|por\s+ciento|pesos)\b|\b\d{3}[\.,]\d{3}\b",
    # Compromiso con fecha
    "fecha": (r"\b\d{1,2}\s+de\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|"
              r"septiembre|octubre|noviembre|diciembre)\b|\bel\s+\d{1,2}\b|"
              r"\b(hoy|mañana|quincena|fin de mes)\b"),
    # Presión: consecuencia legal o urgencia artificial
    "presion": (r"proceso legal|proceso jur[ií]dic|embargo|embargar|judicial|demanda|abogado|"
                r"centrales de riesgo|reporte negativo|tiempo limitado|vence hoy|"
                r"[uú]ltima oportunidad|se vence"),
    # Empatía verbalizada
    "empatia": r"\bentiendo\b|\bcomprendo\b|\blamento\b|\bimagino\b|entendemos|entiendo perfectamente",
    # Pedir permiso para avanzar en el guion, en vez de avanzar
    "gate": r"me permite|le parece si|le gustar[ií]a que|puedo explicarle|me autoriza|desea que le|le puedo comentar",
    # Muletillas: huella de habla espontánea
    "muletilla": r"\bpues\b|\blisto\b|\bbueno\b|\bentonces\b|\bo sea\b|\bmire\b",
    # Cierre operativo: cómo y dónde pagar
    "canal_pago": (r"whatsapp|correo|convenio|efecty|corresponsal|baloto|transferencia|"
                   r"consignaci[oó]n|link de pago|sucursal|bancolombia|nequi|daviplata"),
    # Protocolo de apertura (sin depender de nombres, que están censurados)
    "identificacion": (r"le habla|mi nombre es|me comunico|nos comunicamos|le llamo|lo llamo|"
                       r"la llamo|les llamo|estamos llamando|casa de cobranza|del [aá]rea de|"
                       r"en nombre de"),
    "verificacion": (r"confirmarme si (estoy|es)|hablo con|habla con|hablando con|"
                     r"por favor (la|el) se[ñn]or|me valide si es usted|"
                     r"confirmar (su|sus) (identidad|datos|nombre)|por confidencialidad|"
                     r"con qui[eé]n tengo"),
    "motivo": r"motivo de|referente a|por el tema de|su obligaci[oó]n|su deuda|su cr[eé]dito|su cartera|acuerdo de pago",
    # Proxy de resistencia del deudor
    "objecion": (r"no tengo|no puedo|no me alcanza|sin trabajo|desempleado|estoy mal|"
                 r"ya pagu[eé]|no soy|n[uú]mero equivocado|no me interesa|m[aá]s adelante|"
                 r"despu[eé]s|otro d[ií]a|estoy ocupad|no tengo tiempo|ahora no|"
                 r"est[aá] muy alto|muy caro|no conozco|qu[eé] empresa|es estafa|no confio"),
}

RX = {nombre: re.compile(patron) for nombre, patron in PATRONES.items()}
print(len(RX), "patrones compilados")

## El turno como unidad de análisis

Whisper corta el audio en fragmentos cortos (a veces media frase). Medir sobre fragmentos
infla los conteos y distorsiona el ritmo. Antes de calcular nada, se agrupan los segmentos
consecutivos del mismo rol en un **turno**: una intervención continua de una persona.

Sobre los turnos se define la variable central de ritmo, la **latencia de respuesta**: los
segundos de silencio entre el final de un turno y el inicio de la respuesta del otro rol.
Se calcula para los dos roles a propósito: la del deudor funciona como **control**. Si el
deudor responde igual de rápido en los dos grupos, la lentitud de la IA es del agente y no
del canal telefónico.

In [ ]:
def a_turnos(llamada):
    """Agrupa segmentos consecutivos del mismo rol en un solo turno."""
    turnos = []
    for seg in llamada["segments"]:
        if seg["rol"] not in ("AGENTE", "DEUDOR"):
            continue  # INCIERTO / UNKNOWN no se atribuyen a nadie
        if turnos and turnos[-1]["rol"] == seg["rol"]:
            actual = turnos[-1]
            actual["end"] = seg["end"]
            actual["texto"] += " " + seg["text"]
            actual["voz"] += seg["end"] - seg["start"]
        else:
            turnos.append({
                "rol": seg["rol"],
                "start": seg["start"],
                "end": seg["end"],
                "texto": seg["text"],
                "voz": seg["end"] - seg["start"],
            })
    return turnos


def latencias(turnos, rol_que_responde):
    """Silencios (s) antes de cada respuesta de `rol_que_responde`."""
    return [b["start"] - a["end"]
            for a, b in zip(turnos, turnos[1:])
            if b["rol"] == rol_que_responde and a["rol"] != rol_que_responde
            and b["start"] >= a["end"]]

## Funciones de apoyo

In [ ]:
def mediana(valores):
    """Mediana, o vacío si no hay datos (mejor que un 0 que se confunde con un dato real)."""
    return round(st.median(valores), 3) if valores else ""


def cuenta(patron, texto):
    return len(RX[patron].findall(texto.lower()))


def hay(patron, texto):
    return int(bool(RX[patron].search(texto.lower())))


def riqueza_lexica(texto):
    """Palabras distintas / palabras totales. Un guion rígido repite y baja este número."""
    palabras = re.findall(r"[a-záéíóúñü]+", texto.lower())
    return round(len(set(palabras)) / len(palabras), 3) if palabras else ""


def empatia_con_concesion(turnos):
    """Turnos de empatía del agente seguidos de una concesión concreta (cifra o fecha),
    en ese mismo turno o en el siguiente suyo.

    Separa empatía real de empatía decorativa: decir "entiendo" y repetir el guion no es
    lo mismo que decir "entiendo" y mover la fecha o el monto."""
    idx_agente = [i for i, t in enumerate(turnos) if t["rol"] == "AGENTE"]
    n = 0
    for pos, i in enumerate(idx_agente):
        if not RX["empatia"].search(turnos[i]["texto"].lower()):
            continue
        ventana = turnos[i]["texto"]
        if pos + 1 < len(idx_agente):
            ventana += " " + turnos[idx_agente[pos + 1]]["texto"]
        if RX["cifra"].search(ventana.lower()) or RX["fecha"].search(ventana.lower()):
            n += 1
    return n

## Construcción de la fila

Criterios de normalización, para que las comparaciones no midan simplemente "quién habló más":

- Los conteos de contenido se reportan en crudo **y** normalizados (`muletillas_x_100pal`),
  porque las llamadas humanas son más largas.
- Se usan **medianas** dentro de cada llamada: hay un outlier humano de 20 minutos que
  desplaza cualquier promedio.
- `time_to_offer` se guarda en segundos y como fracción de la llamada.

In [ ]:
def construir_fila(grupo, llamada):
    turnos = a_turnos(llamada)
    t_agente = [t for t in turnos if t["rol"] == "AGENTE"]
    t_deudor = [t for t in turnos if t["rol"] == "DEUDOR"]

    txt_agente = " ".join(t["texto"] for t in t_agente)
    txt_deudor = " ".join(t["texto"] for t in t_deudor)
    apertura = " ".join(t["texto"] for t in t_agente[:5])  # protocolo: primeros 5 turnos

    duracion = llamada["duration_sec"]
    pal_agente, pal_deudor = len(txt_agente.split()), len(txt_deudor.split())
    voz_agente = sum(t["voz"] for t in t_agente)
    voz_deudor = sum(t["voz"] for t in t_deudor)
    voz_total = sum(s["end"] - s["start"] for s in llamada["segments"])

    # primer turno del agente que menciona una cifra concreta
    tto = next((t["start"] for t in t_agente if RX["cifra"].search(t["texto"].lower())), None)

    n_empatia = cuenta("empatia", txt_agente)
    n_empatia_conc = empatia_con_concesion(turnos)
    n_muletillas = cuenta("muletilla", txt_agente)
    n_objeciones = cuenta("objecion", txt_deudor)
    sin_rol = sum(1 for s in llamada["segments"] if s["rol"] in ("INCIERTO", "UNKNOWN"))

    return {
        # --- identificación ---
        "short_id": llamada["short_id"],
        "grupo": grupo,
        "duracion_seg": duracion,

        # --- ritmo conversacional ---
        "latencia_agente_med": mediana(latencias(turnos, "AGENTE")),
        "latencia_deudor_med": mediana(latencias(turnos, "DEUDOR")),
        "silencio_pct": round(max(0.0, 1 - voz_total / duracion), 3),
        "n_turnos": len(turnos),
        "turnos_por_min": round(len(turnos) / (duracion / 60), 2),
        "dur_turno_agente_med": mediana([t["voz"] for t in t_agente]),
        "dur_turno_deudor_med": mediana([t["voz"] for t in t_deudor]),
        "vel_agente_pps": round(pal_agente / voz_agente, 2) if voz_agente else "",
        "vel_deudor_pps": round(pal_deudor / voz_deudor, 2) if voz_deudor else "",

        # --- reparto de habla (ver caveat de diarización) ---
        "palabras_agente": pal_agente,
        "palabras_deudor": pal_deudor,
        "share_palabras_agente": (round(pal_agente / (pal_agente + pal_deudor), 3)
                                  if pal_agente + pal_deudor else ""),
        "share_tiempo_agente": (round(voz_agente / (voz_agente + voz_deudor), 3)
                                if voz_agente + voz_deudor else ""),

        # --- conducta del agente ---
        "n_presion_legal": cuenta("presion", txt_agente),
        "n_empatia": n_empatia,
        "n_empatia_con_concesion": n_empatia_conc,
        "tasa_empatia_efectiva": round(n_empatia_conc / n_empatia, 3) if n_empatia else "",
        "n_gates": cuenta("gate", txt_agente),
        "n_preguntas_agente": sum(1 for t in t_agente if "?" in t["texto"]),
        "muletillas_x_100pal": round(100 * n_muletillas / pal_agente, 2) if pal_agente else "",
        "riqueza_lexica_agente": riqueza_lexica(txt_agente),

        # --- oferta y cierre ---
        "menciona_cifra": int(tto is not None),
        "time_to_offer_seg": round(tto, 1) if tto is not None else "",
        "time_to_offer_pct": round(tto / duracion, 3) if tto is not None else "",
        "menciona_canal_pago": hay("canal_pago", txt_agente),

        # --- protocolo de apertura ---
        "se_identifica": hay("identificacion", apertura),
        "verifica_identidad": hay("verificacion", apertura),
        "declara_motivo": hay("motivo", txt_agente),

        # --- conducta del deudor ---
        "n_objeciones_lex": n_objeciones,
        "objeta": int(n_objeciones > 0),

        # --- calidad de la medición / auditabilidad ---
        "n_speakers_detected": llamada["n_speakers_detected"],
        "diarizacion_colapsada": int(llamada["n_speakers_detected"] == 1),
        "diarizacion_limpia": int(llamada["n_speakers_detected"] == 2 and sin_rol == 0),
        "pct_segmentos_sin_rol": (round(sin_rol / len(llamada["segments"]), 3)
                                  if llamada["segments"] else ""),
    }

## Generar y guardar

In [ ]:
BASE = [construir_fila(grupo, llamada) for grupo, llamada in LLAMADAS]
COLUMNAS = list(BASE[0].keys())

csv_path = OUT_DIR / "base_analitica.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNAS)
    writer.writeheader()
    writer.writerows(BASE)

assert len(BASE) == 97, f"Se esperaban 97 llamadas, hay {len(BASE)}"
print(f"{len(BASE)} llamadas x {len(COLUMNAS)} variables -> {csv_path.name}")

## Resumen comparativo

Para variables continuas se muestra la **mediana** (robusta al outlier de 20 minutos); para
las de conteo, el **promedio** y el **porcentaje de llamadas donde el fenómeno aparece al
menos una vez**, que es lo que se puede comunicar sin ambigüedad.

Este cuadro es exploratorio: los contrastes formales (Mann-Whitney y proporciones) van en el
notebook de análisis.

In [ ]:
CONTINUAS = [
    "duracion_seg", "latencia_agente_med", "latencia_deudor_med", "silencio_pct",
    "turnos_por_min", "dur_turno_agente_med", "dur_turno_deudor_med",
    "vel_agente_pps", "share_palabras_agente", "muletillas_x_100pal",
    "riqueza_lexica_agente", "time_to_offer_seg",
]
CONTEOS = [
    "n_presion_legal", "n_empatia", "n_empatia_con_concesion", "n_gates",
    "n_preguntas_agente", "n_objeciones_lex",
]
BINARIAS = [
    "menciona_cifra", "menciona_canal_pago", "se_identifica", "verifica_identidad",
    "declara_motivo", "objeta", "diarizacion_colapsada", "diarizacion_limpia",
]

H = [r for r in BASE if r["grupo"] == "humano"]
IA = [r for r in BASE if r["grupo"] == "ia"]
vals = lambda filas, col: [float(r[col]) for r in filas if r[col] != ""]


def imprimir(titulo, columnas, resumen, encabezado):
    print(f"\n{titulo}")
    print(f"{'variable':<26}{encabezado[0]:>12}{encabezado[1]:>12}{'n h/ia':>10}")
    print("-" * 60)
    for col in columnas:
        h, i = vals(H, col), vals(IA, col)
        print(f"{col:<26}{resumen(h):>12}{resumen(i):>12}{f'{len(h)}/{len(i)}':>10}")


fmt = lambda x: f"{x:.2f}"
imprimir("MEDIANAS", CONTINUAS, lambda v: fmt(st.median(v)), ("humano", "IA"))
imprimir("CONTEOS (promedio por llamada)", CONTEOS, lambda v: fmt(st.mean(v)), ("humano", "IA"))
imprimir("PRESENCIA (% de llamadas)", BINARIAS,
         lambda v: f"{100 * st.mean(v):.0f}%", ("humano", "IA"))

## Qué queda pendiente

1. **Latencia sobre diarización limpia.** En 29 % de las llamadas humanas la diarización
   colapsó a un hablante, y ahí los cambios de rol vienen del LLM, no del audio. La
   diferencia de latencia debe recalcularse sobre `diarizacion_limpia == 1` antes de
   reportarse como hallazgo.
2. **`share_palabras_agente` está contaminado** en las llamadas humanas: la diarización le
   atribuyó al agente turnos cortos del deudor. No usar como métrica de titular sin
   restringir al subconjunto limpio.
3. **Variables de resultado.** Compromiso de pago, tipo de objeción y manejo de la objeción
   requieren etiquetado semántico con LLM: son el insumo que falta para responder quién es
   más efectivo.